Ten skrypt konfiguruje i instrumentuje agenta Smolagents przy użyciu Langfuse, a następnie uruchamia go na każdym przykładzie ze zbioru GSM8K, zbierając szczegółowe ślady działania. Po zakończeniu obliczeń agent wysyła zarejestrowane dane i oceny (score) do usługi Langfuse.

# Setup

In [ ]:
!uv pip install -qU 'smolagents[telemetry]' opentelemetry-sdk opentelemetry-exporter-otlp openinference-instrumentation-smolagents langfuse datasets duckduckgo_search markdownify requests huggingface_hub fsspec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.0/119.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.8/65.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.9/194.9 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.4/275.4 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 103.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.2/299.2 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.2/125.2 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

*   `smolagents[telemetry]`: Instaluje lub aktualizuje pakiet `smolagents`.  Dodatkowe nawiasy kwadratowe `[telemetry]` wskazują, że instalowana jest wersja pakietu zawierająca dodatkowe funkcjonalności związane z telemetrycznymi danymi.
*   `opentelemetry-sdk`: Instaluje bibliotekę OpenTelemetry SDK (Software Development Kit).  OpenTelemetry to zestaw narzędzi do generowania, zbierania i eksportowania danych telemetrycznych (metryk, logów, śladów).
*   `opentelemetry-exporter-otlp`: Instaluje komponent OpenTelemetry odpowiedzialny za eksportowanie danych telemetrycznych przy użyciu protokołu OTLP (OpenTelemetry Protocol).  OTLP jest standardowym protokołem do przesyłania danych telemetrycznych.
*   `openinference-instrumentation-smolagents`: Instaluje bibliotekę instrumentacji dla SmolAgents, która umożliwia monitorowanie i śledzenie działania modeli wnioskowania wykorzystywanych przez agentów Smol.

In [ ]:
# Standard library imports
import base64
import os

# Third-party library imports
from datasets import load_dataset
from google.colab import userdata
from langfuse import Langfuse
import pandas as pd

# Opentelemetry imports
from opentelemetry import trace
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.trace import format_trace_id

# Project-specific imports
from openinference.instrumentation.smolagents import SmolagentsInstrumentor
from smolagents import (
    CodeAgent,
    HfApiModel,
)

Ten kod zawiera sekcję importów niezbędnych bibliotek do działania programu. Biblioteki zostały podzielone na trzy kategorie: standardowe biblioteki Pythona, biblioteki zewnętrzne oraz biblioteki związane z OpenTelemetry.

*   **Standard library imports:**
    *   `base64`:  Umożliwia kodowanie i dekodowanie danych w formacie Base64.
    *   `os`: Zapewnia dostęp do funkcjonalności systemu operacyjnego, takich jak praca z plikami i zmiennymi środowiskowymi.

*   **Third-party library imports:**
    *   `datasets`: Biblioteka Hugging Face `datasets`, służąca do łatwego pobierania i manipulowania zbiorami danych.
    *   `google.colab.userdata`:  Pozwala na dostęp do sekretów (np. kluczy API) przechowywanych w środowisku Google Colab.
    *   `langfuse`: Biblioteka Langfuse, platformy do monitoringu i debugowania aplikacji LLM.
    *   `pandas`: Biblioteka `pandas`, używana do analizy danych i pracy z tabelami (DataFrames).

*   **Opentelemetry imports:**
    *   `opentelemetry.trace`:  Moduł OpenTelemetry odpowiedzialny za śledzenie wykonywania kodu.
    *   `opentelemetry.exporter.otlp.proto.http.trace_exporter`: Eksportuje dane śledzenia do systemu backendowego przy użyciu protokołu OTLP (OpenTelemetry Protocol) przez HTTP.
    *   `opentelemetry.sdk.trace`:  Zawiera klasy i funkcje potrzebne do konfiguracji i uruchomienia OpenTelemetry SDK dla śledzenia.
    *   `opentelemetry.sdk.trace.export`: Moduł odpowiedzialny za eksport danych śledzenia.
    *   `opentelemetry.trace.format_trace_id`: Funkcja formatująca identyfikator śledzenia.

*   **Project-specific imports:**
    *   `openinference.instrumentation.smolagents`:  Moduł instrumentacji SmolAgents dla OpenInference, umożliwiający monitorowanie działania agentów.
    *   `smolagents`: Biblioteka SmolAgents zawierająca klasy i funkcje do tworzenia inteligentnych agentów:
        *   `CodeAgent`: Agent potrafiący generować i wykonywać kod.
        *   `DuckDuckGoSearchTool`: Narzędzie umożliwiające wyszukiwanie informacji w DuckDuckGo.
        *   `HfApiModel`: Klasa reprezentująca model językowy z Hugging Face Hub.
        *   `LiteLLMModel`:  Klasa reprezentująca lekki model językowy (dodana później).
        *   `ToolCallingAgent`: Agent potrafiący korzystać z narzędzi.
        *   `VisitWebpageTool`: Narzędzie umożliwiające odwiedzanie stron internetowych i pobieranie ich zawartości.



Ten kod przygotowuje środowisko programistyczne poprzez zaimportowanie wszystkich niezbędnych bibliotek, które będą wykorzystywane w dalszej części programu do tworzenia agentów, interakcji z modelami językowymi, monitorowania działania aplikacji oraz zbierania danych telemetrycznych.

In [ ]:
class CFG:
    model = "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B"

In [ ]:
os.environ["LANGFUSE_PUBLIC_KEY"] = userdata.get("langfuse_pub")
os.environ["LANGFUSE_SECRET_KEY"] = userdata.get("langfuse_prv")
os.environ["LANGFUSE_HOST"] = "https://cloud.langfuse.com"  #

LANGFUSE_AUTH = base64.b64encode(
    f"{os.environ.get('LANGFUSE_PUBLIC_KEY')}:{os.environ.get('LANGFUSE_SECRET_KEY')}".encode()
).decode()

os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = (
    os.environ.get("LANGFUSE_HOST") + "/api/public/otel"
)
os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = f"Authorization=Basic {LANGFUSE_AUTH}"

os.environ["OPENAI_API_KEY"] = userdata.get("openaivision")

Ten kod konfiguruje zmienne środowiskowe niezbędne do działania aplikacji, w szczególności integrując ją z platformą Langfuse i OpenTelemetry. Wykorzystuje dane uwierzytelniające przechowywane w Google Colab userdata.

*   **Konfiguracja kluczy Langfuse:**
    *   `os.environ["LANGFUSE_PUBLIC_KEY"] = userdata.get('langfuse_pub')`: Pobiera publiczny klucz API Langfuse z userdata i ustawia go jako zmienną środowiskową `LANGFUSE_PUBLIC_KEY`.
    *   `os.environ["LANGFUSE_SECRET_KEY"] = userdata.get('langfuse_prv')`: Pobiera prywatny klucz API Langfuse z userdata i ustawia go jako zmienną środowiskową `LANGFUSE_SECRET_KEY`.
    *   `os.environ["LANGFUSE_HOST"] = "https://cloud.langfuse.com" # 🇪🇺 EU region`: Ustawia adres hosta Langfuse na `https://cloud.langfuse.com`, wskazując na serwery w regionie europejskim (oznaczone flagą UE).

*   **Generowanie nagłówka autoryzacji dla Langfuse:**
    *   `LANGFUSE_AUTH = base64.b64encode(f"{os.environ.get('LANGFUSE_PUBLIC_KEY')}:{os.environ.get('LANGFUSE_SECRET_KEY')}".encode()).decode()`:  Łączy publiczny i prywatny klucz API Langfuse za pomocą dwukropka, koduje wynik w Base64, a następnie dekoduje go do ciągu znaków. Ten ciąg jest przechowywany w zmiennej `LANGFUSE_AUTH` i będzie używany jako nagłówek autoryzacji przy komunikacji z Langfuse.

*   **Konfiguracja OpenTelemetry:**
    *   `os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = os.environ.get("LANGFUSE_HOST") + "/api/public/otel"`: Ustawia adres punktu końcowego OTLP (OpenTelemetry Protocol) na serwer Langfuse, dodając ścieżkę `/api/public/otel` do adresu hosta Langfuse.  Oznacza to, że dane telemetryczne będą wysyłane bezpośrednio do Langfuse.
    *   `os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = f"Authorization=Basic {LANGFUSE_AUTH}"`: Ustawia nagłówki HTTP dla eksportu OTLP, dodając nagłówek `Authorization` z wartością `Basic` oraz wygenerowanym wcześniej ciągiem Base64 (`LANGFUSE_AUTH`).  Ten nagłówek zapewnia uwierzytelnienie przy wysyłaniu danych telemetrycznych do Langfuse.

*   **Konfiguracja klucza OpenAI:**
    *   `os.environ["OPENAI_API_KEY"] = userdata.get('openaivision')`: Pobiera klucz API OpenAI z userdata i ustawia go jako zmienną środowiskową `OPENAI_API_KEY`.



Podsumowując, ten kod pobiera dane uwierzytelniające z Google Colab userdata, konfiguruje zmienne środowiskowe dla Langfuse i OpenTelemetry oraz przygotowuje nagłówek autoryzacji. Dzięki temu aplikacja może komunikować się z platformą Langfuse w celu monitorowania i debugowania oraz wysyłać dane telemetryczne do analizy. Dodatkowo, ustawia klucz API OpenAI umożliwiając korzystanie z modeli OpenAI.

In [ ]:
trace_provider = TracerProvider()
trace_provider.add_span_processor(SimpleSpanProcessor(OTLPSpanExporter()))

SmolagentsInstrumentor().instrument(tracer_provider=trace_provider)


trace.set_tracer_provider(trace_provider)


tracer = trace.get_tracer("my.tracer.name")

# Test: czy to działa na danych?

In [ ]:
dataset = load_dataset("openai/gsm8k", "main", split="test")
df = pd.DataFrame(dataset)
print("First few rows of GSM8K dataset:")
print(df.head())

README.md:   0%|          | 0.00/7.94k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

First few rows of GSM8K dataset:
                                            question  \
0  Janet’s ducks lay 16 eggs per day. She eats th...   
1  A robe takes 2 bolts of blue fiber and half th...   
2  Josh decides to try flipping a house.  He buys...   
3  James decides to run 3 sprints 3 times a week....   
4  Every day, Wendi feeds each of her chickens th...   

                                              answer  
0  Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eg...  
1  It takes 2/2=<<2/2=1>>1 bolt of white fiber\nS...  
2  The cost of the house and repairs came out to ...  
3  He sprints 3*3=<<3*3=9>>9 times\nSo he runs 9*...  
4  If each chicken eats 3 cups of feed per day, t...  


In [ ]:
langfuse = Langfuse()

langfuse_dataset_name = "gsm8k_dataset_huggingface"

# Create a dataset in Langfuse
langfuse.create_dataset(
    name=langfuse_dataset_name,
    description="GSM8K benchmark dataset uploaded from Huggingface",
    metadata={"date": "2025-05-01", "type": "benchmark"},
)

Dataset(id='cmatmv7hn00r2ad075io3nagz', name='gsm8k_dataset_huggingface', description='GSM8K benchmark dataset uploaded from Huggingface', metadata={'date': '2025-05-01', 'type': 'benchmark'}, project_id='cmam55kmo0011ad073ozxrz6h', created_at=datetime.datetime(2025, 5, 18, 12, 29, 20, 171000, tzinfo=datetime.timezone.utc), updated_at=datetime.datetime(2025, 5, 18, 12, 36, 29, 898000, tzinfo=datetime.timezone.utc))

Ten kod tworzy nowy zbiór danych w Langfuse, platformie do monitorowania i debugowania aplikacji LLM. Zbiór danych reprezentuje dane ze zbioru GSM8K załadowanego z Hugging Face Datasets.

*   **`langfuse = Langfuse()`**: Tworzy instancję klasy `Langfuse`, inicjalizując połączenie z platformą Langfuse.  Zakłada się, że klucze API Langfuse zostały już skonfigurowane jako zmienne środowiskowe (jak w poprzednich fragmentach kodu).

*   **`langfuse_dataset_name = "gsm8k_dataset_huggingface"`**: Definiuje nazwę zbioru danych w Langfuse. Nazwa ta będzie używana do identyfikacji i odwoływania się do tego zbioru danych w interfejsie Langfuse.

*   **`langfuse.create_dataset(...)`**: Wywołuje metodę `create_dataset` obiektu `langfuse`, aby utworzyć nowy zbiór danych w Langfuse.
    *   `name=langfuse_dataset_name`: Ustawia nazwę zbioru danych na wartość zmiennej `langfuse_dataset_name`.
    *   `description="GSM8K benchmark dataset uploaded from Huggingface"`: Dodaje opis do zbioru danych, wyjaśniając jego pochodzenie i cel.
    *   `metadata={...}`:  Dodaje metadane do zbioru danych w postaci słownika. Metadane zawierają dodatkowe informacje o zbiorze danych:
        *   `"date": "2025-05-01"`: Data utworzenia lub załadowania zbioru danych.
        *   `"type": "benchmark"`: Typ zbioru danych, w tym przypadku zbiór benchmarkowy.

W efekcie tego kodu nowy zbiór danych o nazwie "gsm8k\_dataset\_huggingface" jest tworzony w Langfuse, zawierający informacje o zbiorze GSM8K załadowanym z Hugging Face Datasets oraz metadane dotyczące daty i typu zbioru danych.  Ten zbiór danych będzie mógł być wykorzystywany do przechowywania i analizowania wyników działania agentów Smol na zadaniach z tego zbioru.

In [ ]:
for idx, row in df.iterrows():
    langfuse.create_dataset_item(
        dataset_name=langfuse_dataset_name,
        input={"text": row["question"]},
        expected_output={"text": row["answer"]},
        metadata={"source_index": idx},
    )
    # tylko podzbiór - czas!
    if idx >= 2:
        break

Ten kod iteruje po wierszach ramki danych `df` (zawierającej zbiór danych GSM8K) i tworzy oddzielne elementy zbioru danych w Langfuse dla każdego wiersza.

*   **`for idx, row in df.iterrows():`**: Rozpoczyna pętlę iterującą po każdym wierszu ramki danych `df`.
    *   `idx`: Indeks bieżącego wiersza.
    *   `row`: Obiekt Series reprezentujący bieżący wiersz danych.

*   **`langfuse.create_dataset_item(...)`**: Wywołuje metodę `create_dataset_item` obiektu `langfuse`, aby utworzyć nowy element zbioru danych w Langfuse dla każdego wiersza z ramki danych.
    *   `dataset_name=langfuse_dataset_name`: Określa nazwę zbioru danych, do którego ma zostać dodany nowy element (w tym przypadku "gsm8k\_dataset\_huggingface").
    *   `input={"text": row["question"]}`: Definiuje dane wejściowe dla elementu. W tym przypadku jest to pytanie z wiersza `row`, przechowywane w kolumnie "question".  Dane wejściowe są przekazywane jako słownik z kluczem "text".
    *   `expected_output={"text": row["answer"]}`: Definiuje oczekiwane wyjście dla elementu. Jest to odpowiedź na pytanie z wiersza `row`, przechowywana w kolumnie "answer". Dane wyjściowe są przekazywane jako słownik z kluczem "text".
    *   `metadata={"source_index": idx}`: Dodaje metadane do elementu, zawierające indeks źródłowego wiersza w ramce danych `df`.

*   **`if idx >= 9:`**: Sprawdza, czy indeks bieżącego wiersza jest większy lub równy 9.
*   **`break`**: Jeśli warunek z poprzedniego kroku jest spełniony (indeks jest większy lub równy 9), pętla zostaje przerwana. Oznacza to, że tylko pierwsze 10 elementów zbioru danych (od indeksu 0 do 9) zostanie dodanych do Langfuse.  Jest to zrobione w celu ograniczenia czasu wykonania i przetestowania integracji z Langfuse na mniejszym podzbiorze danych.

Podsumowując, kod ten iteruje po zbiorze danych GSM8K, tworząc dla każdego pytania i odpowiedzi oddzielny element w zbiorze danych Langfuse. Dodatkowo, dodaje metadane wskazujące indeks oryginalnego wiersza w ramce danych. Pętla jest przerwana po przetworzeniu pierwszych 10 elementów.

In [ ]:
model = HfApiModel()

agent = CodeAgent(tools=[], model=model, add_base_tools=True)


def run_smolagent(question):
    with tracer.start_as_current_span("Smolagent-Trace") as span:
        span.set_attribute("langfuse.tag", "dataset-run")
        output = agent.run(question)

        current_span = trace.get_current_span()
        span_context = current_span.get_span_context()
        trace_id = span_context.trace_id
        formatted_trace_id = format_trace_id(trace_id)

        langfuse_trace = langfuse.trace(
            id=formatted_trace_id, input=question, output=output
        )
    return langfuse_trace, output

/usr/local/lib/python3.11/dist-packages/smolagents/models.py:1325: FutureWarning: HfApiModel was renamed to InferenceClientModel in version 1.14.0 and will be removed in 1.17.0.
  warnings.warn(


Ten kod definiuje funkcję `run_smolagent`, która uruchamia agenta Smol, śledzi jego działanie za pomocą OpenTelemetry i wysyła informacje o wykonaniu do Langfuse.

*   **`model = HfApiModel()`**: Tworzy instancję modelu językowego `HfApiModel`.
*   **`agent = CodeAgent(...)`**: Tworzy agenta `CodeAgent`, który potrafi generować i wykonywać kod.
    *   `tools=[]`: Określa, że agent nie ma dostępu do żadnych narzędzi (poza tymi dodanymi przez `add_base_tools`).
    *   `model=model`: Przypisuje wcześniej utworzony model językowy do agenta.
    *   `add_base_tools=True`: Dodaje podstawowe narzędzia do agenta, takie jak dostęp do internetu i wykonywanie kodu Pythona.

*   **`def run_smolagent(question):`**: Definiuje funkcję `run_smolagent`, która przyjmuje pytanie jako argument i zwraca informacje o śledzeniu w Langfuse oraz wynik działania agenta.

*   **`with tracer.start_as_current_span("Smolagent-Trace") as span:`**: Rozpoczyna nowy zakres (span) w ramach śledzenia OpenTelemetry, o nazwie "Smolagent-Trace". Blok `with` zapewnia automatyczne zakończenie zakresu po wykonaniu kodu wewnątrz niego. Zmienna `span` reprezentuje bieżący zakres śledzenia.
*   **`span.set_attribute("langfuse.tag", "dataset-run")`**: Dodaje atrybut do zakresu, oznaczający uruchomienie agenta jako część operacji na zbiorze danych.

*   **`output = agent.run(question)`**: Uruchamia agenta z podanym pytaniem i zapisuje wynik w zmiennej `output`.
*   **`current_span = trace.get_current_span()`**: Pobiera bieżący zakres śledzenia OpenTelemetry.
*   **`span_context = current_span.get_span_context()`**: Pobiera kontekst zakresu, który zawiera informacje o śledzeniu.
*   **`trace_id = span_context.trace_id`**: Pobiera identyfikator śledzenia z kontekstu zakresu.
*   **`formatted_trace_id = format_trace_id(trace_id)`**: Formatuje identyfikator śledzenia do postaci tekstowej.

*   **`langfuse_trace = langfuse.trace(...)`**: Wysyła informacje o śledzeniu do Langfuse.
    *   `id=formatted_trace_id`: Ustawia identyfikator śledzenia w Langfuse na sformatowany identyfikator z OpenTelemetry.
    *   `input=question`: Przekazuje pytanie jako dane wejściowe do Langfuse.
    *   `output=output`: Przekazuje wynik działania agenta jako wyjście do Langfuse.

*   **`return langfuse_trace, output`**: Zwraca obiekt śledzenia z Langfuse oraz wynik działania agenta.

Podsumowując, funkcja `run_smolagent` uruchamia agenta Smol, śledzi jego działanie za pomocą OpenTelemetry i wysyła informacje o śledzeniu (wraz z danymi wejściowymi i wyjściowymi) do Langfuse w celu monitorowania i analizy.

In [ ]:
dataset = langfuse.get_dataset(langfuse_dataset_name)

# Run our agent against each dataset item (limited to first 10 above)
for item in dataset.items:
    langfuse_trace, output = run_smolagent(item.input["text"])

    # Link the trace to the dataset item for analysis
    item.link(
        langfuse_trace,
        run_name="smolagent-notebook-run-01",
        run_metadata={"model": model.model_id},
    )

    # store a quick evaluation score for demonstration
    langfuse_trace.score(name="<example_eval>", value=1, comment="This is a comment")

# Flush data to ensure all telemetry is sent
langfuse.flush()

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Josh decides to try flipping a house.  He buys a house for $80,000 and then puts in $50,000 in repairs.  This   │
│ increased the value of the house by 150%.  How much profit did he make?                                         │
│                                                                                                                 │
╰─ HfApiModel - Qwen/Qwen2.5-Coder-32B-Instruct ──────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  purchase_price = 80000                                                                                           
  repair_cost = 50000                                                                                              
  total_cost = purchase_price + repair_cost                                                                        
                                                                                                                   
  # Increase in value is 150% of the original purchase price                                                       
  value_increase = 150 / 100 * purchase_price                                                                      
                                                                                                                   
  # New value of the house                                                                                         
  new_value = purchase_price + value_increase                                                                      
                                                                                                                   
  # Profit calculation                                                                                             
  profit = new_value - total_cost                                                                                  
  final_answer(profit)                                                                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out - Final answer: 70000.0

[Step 1: Duration 0.98 seconds| Input tokens: 2,147 | Output tokens: 157]

...

[Step 1: Duration 0.32 seconds| Input tokens: 2,155 | Output tokens: 251]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Total eggs laid per day                                                                                        
  total_eggs = 16                                                                                                  
                                                                                                                   
  # Eggs eaten for breakfast                                                                                       
  eggs_eaten = 3                                                                                                   
                                                                                                                   
  # Eggs used for baking muffins                                                                                   
  eggs_baked = 4                                                                                                   
                                                                                                                   
  # Price per egg                                                                                                  
  price_per_egg = 2                                                                                                
                                                                                                                   
  # Calculate the remaining eggs after breakfast and baking                                                        
  remaining_eggs = total_eggs - eggs_eaten - eggs_baked                                                            
                                                                                                                   
  # Calculate the earnings from selling the remaining eggs                                                         
  earnings_per_day = remaining_eggs * price_per_egg                                                                
                                                                                                                   
  print(earnings_per_day)                                                                                          
  final_answer(earnings_per_day)                                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
18

Out - Final answer: 18

[Step 2: Duration 0.25 seconds| Input tokens: 4,743 | Output tokens: 445]

Podsumowując, kod ten uruchamia agenta Smol na każdym elemencie zbioru danych w Langfuse, łączy wyniki śledzenia z odpowiednimi elementami zbioru danych, dodaje ocenę do każdego śledzenia i wysyła wszystkie dane telemetryczne do Langfuse.  Pozwala to na kompleksową analizę działania agenta na zbiorze danych GSM8K.

https://cloud.langfuse.com/project/cmam55kmo0011ad073ozxrz6h/traces